# Huấn luyện mô hình nhận diện ô đỗ trống / có xe

**Bài toán.** Camera nhìn cố định xuống bãi, vị trí 12 ô đã biết (ROI trong `pc_server/config/layout.json`).
Vì vậy không cần mô hình *phát hiện vật thể* (YOLO) đi tìm xe khắp ảnh: chỉ cần cắt từng ô và **phân loại nhị phân**
`empty` / `occupied`. Cách này nhẹ hơn, chính xác hơn với ô bị che một phần, không phụ thuộc xe có "giống xe thật" hay không,
và chạy 12 ô trong một batch.

**Dữ liệu: CNRPark+EXT** (Amato et al., 2017) — ~157 nghìn ảnh crop từng ô, 150×150, gán nhãn tay.
- **CNR-EXT**: 9 camera, nhiều góc nhìn, 3 kiểu thời tiết (nắng / âm u / mưa), có bóng cây, xe che khuất.
  Split chuẩn của tác giả **chia theo ngày** (train / val / test là các ngày khác nhau) → đánh giá không bị rò rỉ.
- **CNRPark A/B**: 2 camera khác, **không dùng khi train** → dùng để đo khả năng tổng quát sang camera mới.

So với PKLot (3 bãi, góc nhìn ít hơn, ảnh gần như trùng lặp giữa các khung), CNRPark+EXT đa dạng hơn về góc nhìn và che khuất,
gần với tình huống camera tự lắp hơn.

**Quy trình notebook**
1. Khám phá dữ liệu
2. Sàng lọc 5 kiến trúc (mAlexNet của bài báo gốc, MobileNetV3-S/L, EfficientNet-B0, ResNet18) theo độ chính xác **và** tốc độ CPU
3. Tối ưu siêu tham số bằng Optuna (TPE + cắt tỉa thử nghiệm kém)
4. Huấn luyện cuối trên toàn bộ tập train (AMP, AdamW, cosine + warmup, EMA, early stopping)
5. Đánh giá đầy đủ: metric, ma trận nhầm lẫn, ROC/PR, hiệu chỉnh xác suất, theo thời tiết/camera, camera chưa thấy,
   độ bền với nhiễu, phân tích lỗi, Grad-CAM, tốc độ
6. Tinh chỉnh với ảnh xe mô hình tự thu (nếu có)
7. Xuất mô hình cho `pc_server` (`DETECTOR=classifier`)

> Chạy `python download_dataset.py` rồi `python build_cache.py` trước khi chạy notebook.
> Trên Kaggle: không cần bước này, ô đầu tiên tự tải dữ liệu (xem README, mục *Train trên Kaggle*).

In [ ]:
# Kaggle: chép code từ Dataset đính kèm vào /tmp/parking, tải CNRPark+EXT (~500 MB, cần bật Internet) và làm việc tại đó.
# Kết quả (results/, models/) ghi thẳng vào /kaggle/working → tab Output. Chạy trên máy cá nhân: ô này không làm gì.
import os, sys, shutil
from pathlib import Path

KAGGLE = Path("/kaggle/input").is_dir()
if KAGGLE:
    WORKDIR = Path("/tmp/parking"); WORKDIR.mkdir(exist_ok=True)
    code = next(Path("/kaggle/input").rglob("parking_lib.py")).parent
    for f in ("parking_lib.py", "download_dataset.py"):
        shutil.copy(code / f, WORKDIR / f)
    os.chdir(WORKDIR); sys.path.insert(0, str(WORKDIR))

    # ảnh xe mô hình tự thu (nếu có): thư mục bất kỳ trong /kaggle/input chứa empty/ và occupied/
    custom = next((d.parent for d in Path("/kaggle/input").rglob("occupied") if (d.parent / "empty").is_dir()), None)
    if custom and not (WORKDIR / "data" / "custom").exists():
        shutil.copytree(custom, WORKDIR / "data" / "custom")
        print("Dữ liệu tự thu:", custom)

    if not (WORKDIR / "data" / "CNR-EXT" / "LABELS").is_dir():
        import download_dataset as dd
        dd.DATA.mkdir(exist_ok=True)
        for name, folder in dd.FILES.items():
            dd.extract(dd.download(name), folder)
        shutil.rmtree(dd.DATA / "raw", ignore_errors=True)  # bỏ file zip cho đỡ tốn đĩa

    try:
        import optuna
    except ImportError:
        import subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "optuna"], check=True)
print("Kaggle:", KAGGLE, "| thư mục làm việc:", Path.cwd())

In [ ]:
import os, sys, json, time, math, random, copy, gc, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, precision_recall_curve

import parking_lib as pl

warnings.filterwarnings("ignore", category=UserWarning)
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})

ROOT = Path.cwd()
DATA = ROOT / "data"
OUT = Path("/kaggle/working") if KAGGLE else ROOT  # trên Kaggle chỉ /kaggle/working được giữ lại sau khi chạy
RESULTS = OUT / "results"; RESULTS.mkdir(exist_ok=True)
EXPORT = (ROOT.parent / "pc_server" / "models") if (ROOT.parent / "pc_server").is_dir() else (OUT / "models")  # không có pc_server cạnh đây thì xuất vào models/
EXPORT.mkdir(exist_ok=True)

CFG = dict(
    img_size=128, batch=256, workers=min(8, os.cpu_count()), seed=42,
    screen_train=20000, screen_val=6000, screen_epochs=3,   # sàng lọc kiến trúc
    trials=14, trial_epochs=3,                               # Optuna
    final_epochs=15, patience=4, ema_decay=0.999,            # huấn luyện cuối
    latency_budget_ms=50,                                    # 12 ô / khung hình trên CPU
)
# SMOKE=1: chạy thử toàn bộ notebook trong vài phút với dữ liệu rất nhỏ (để kiểm tra lỗi, KHÔNG dùng kết quả)
SMOKE = os.getenv("SMOKE") == "1"
if SMOKE:
    CFG.update(screen_train=1024, screen_val=512, screen_epochs=1, trials=2, trial_epochs=1, final_epochs=1)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_all(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "", "| torch", torch.__version__)

def savefig(name):
    plt.savefig(RESULTS / f"{name}.png", bbox_inches="tight")

## 1. Dữ liệu

In [ ]:
CNR = DATA / "CNR-EXT"
train_df = pl.load_cnr_split(CNR, CNR / "LABELS" / "train.txt")
val_df   = pl.load_cnr_split(CNR, CNR / "LABELS" / "val.txt")
test_df  = pl.load_cnr_split(CNR, CNR / "LABELS" / "test.txt")
ab_df    = pl.load_cnrpark_ab(DATA / "CNRPark")
if SMOKE:
    train_df, val_df, test_df, ab_df = [d.sample(n, random_state=0).reset_index(drop=True)
                                        for d, n in [(train_df, 3000), (val_df, 1500), (test_df, 4500), (ab_df, 1000)]]
store = pl.JpegStore(DATA / "cache") if (DATA / "cache" / "index.npz").exists() else None
print("Dùng cache gói ảnh:", store is not None)

splits = {"train": train_df, "val": val_df, "test": test_df, "CNRPark A/B (camera mới)": ab_df}
summary = pd.DataFrame({k: d.label.value_counts().rename({0: "empty", 1: "occupied"}) for k, d in splits.items()}).T
summary["total"] = summary.sum(axis=1)
summary["% occupied"] = (summary.occupied / summary.total * 100).round(1)
display(summary)

# kiểm tra rò rỉ: các split không được trùng ngày chụp
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    print(f"Số ngày trùng {a}/{b}:", len(set(splits[a].date) & set(splits[b].date)))

In [ ]:
all_df = pd.concat([d.assign(split=k) for k, d in list(splits.items())[:3]])
all_df["class"] = all_df.label.map({0: "empty", 1: "occupied"})
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=all_df, x="split", hue="class", ax=ax[0], palette=["#22c55e", "#ef4444"]); ax[0].set_title("Số ảnh theo split")
sns.countplot(data=all_df, x="weather", hue="class", ax=ax[1], palette=["#22c55e", "#ef4444"]); ax[1].set_title("Theo thời tiết")
sns.countplot(data=all_df, x="camera", hue="class", ax=ax[2], palette=["#22c55e", "#ef4444"],
              order=sorted(all_df.camera.unique())); ax[2].set_title("Theo camera"); ax[2].tick_params(axis="x", rotation=45)
plt.tight_layout(); savefig("eda_counts"); plt.show()

In [ ]:
viewer = pl.PatchDataset(test_df, None, store)
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
rng = np.random.default_rng(1)
for r, (lab, w) in enumerate([(0, None), (1, None), (None, "RAINY")]):
    sub = test_df if lab is None else test_df[test_df.label == lab]
    if w: sub = sub[sub.weather == w]
    for c, i in enumerate(rng.choice(sub.index, 8, replace=False)):
        j = test_df.index.get_loc(i)
        axes[r, c].imshow(viewer.load(j)); axes[r, c].axis("off")
        axes[r, c].set_title(f"{'occupied' if test_df.label[i] else 'empty'}\n{test_df.camera[i]} {test_df.weather[i][:4]}", fontsize=8)
plt.suptitle("Ví dụ: hàng 1 ô trống, hàng 2 có xe, hàng 3 trời mưa"); plt.tight_layout(); savefig("eda_samples"); plt.show()

### Tăng cường dữ liệu
Ảnh từ Pi Camera (xe mô hình, nhìn từ trên, ánh sáng trong phòng) **khác xa** ảnh bãi xe thật. Tăng cường mạnh giúp mô hình
học đặc trưng "có vật thể chiếm chỗ" thay vì học thuộc màu/góc của camera cụ thể: cắt-phóng ngẫu nhiên, lật, xoay/nghiêng nhẹ,
đổi màu/độ sáng, ảnh xám, làm mờ, xoá ngẫu nhiên một mảng (mô phỏng che khuất).
Mức tăng cường này được **cố định** (không để Optuna chọn) vì tập val cùng camera với train nên sẽ luôn "thưởng" cho tăng cường yếu,
trong khi mục tiêu thật là tổng quát sang camera mới.

In [ ]:
train_tf, eval_tf = pl.build_transforms(CFG["img_size"], strong=True)
img = viewer.load(test_df.index.get_loc(test_df[test_df.label == 1].index[3]))
fig, axes = plt.subplots(1, 9, figsize=(16, 2.2))
axes[0].imshow(img); axes[0].set_title("gốc", fontsize=9)
for a in axes[1:]:
    a.imshow(pl.denorm(train_tf(img)))
for a in axes: a.axis("off")
plt.tight_layout(); savefig("augmentations"); plt.show()

In [ ]:
def loader(df, tf, bs=CFG["batch"], shuffle=False, workers=CFG["workers"]):
    return DataLoader(pl.PatchDataset(df, tf, store), batch_size=bs, shuffle=shuffle, num_workers=workers,
                      pin_memory=device.type == "cuda", persistent_workers=workers > 0, drop_last=shuffle)


def fit(arch, hp, train_loader, val_loader, epochs, trial=None, ema=False, verbose=True, pretrained=True, init=None):
    # Huấn luyện, giữ trọng số có val loss thấp nhất, dừng sớm sau CFG['patience'] epoch không cải thiện.
    model = pl.build_model(arch, dropout=hp["dropout"], pretrained=pretrained).to(device)
    if init is not None:
        model.load_state_dict(init)
    model = model.to(memory_format=torch.channels_last)
    opt = pl.make_optimizer(model, hp["lr"], hp["weight_decay"])
    steps = epochs * len(train_loader)
    sch = pl.cosine_with_warmup(opt, steps, max(1, int(steps * hp.get("warmup", 0.05))))
    scaler = torch.amp.GradScaler(enabled=device.type == "cuda")
    ema_m = pl.EMA(model, CFG["ema_decay"]) if ema else None
    hist, best, bad = [], None, 0
    for ep in range(epochs):
        t0 = time.time()
        tl, ta = pl.train_one_epoch(model, train_loader, opt, sch, scaler, device, hp["label_smoothing"], ema_m)
        evm = ema_m.model if ema_m else model
        p, y, vl = pl.predict(evm, val_loader, device)
        m = pl.binary_metrics(y, p)
        hist.append(dict(epoch=ep + 1, train_loss=tl, train_acc=ta, val_loss=vl, val_acc=m["accuracy"],
                         val_f1=m["f1"], val_auc=m["roc_auc"], lr=opt.param_groups[0]["lr"], sec=time.time() - t0))
        if verbose:
            h = hist[-1]
            print(f"ep {h['epoch']:>2} | train loss {tl:.4f} acc {ta:.4f} | val loss {vl:.4f} acc {h['val_acc']:.4f} "
                  f"f1 {h['val_f1']:.4f} auc {h['val_auc']:.4f} | {h['sec']:.0f}s")
        if best is None or vl < best[0]:
            best, bad = (vl, copy.deepcopy(evm.state_dict()), ep + 1), 0
        else:
            bad += 1
        if trial is not None:
            trial.report(vl, ep)
            if trial.should_prune():
                raise optuna.TrialPruned()
        if bad >= CFG["patience"]:
            if verbose: print("Dừng sớm.")
            break
    evm.load_state_dict(best[1])
    return evm, pd.DataFrame(hist), best[2]

## 2. Sàng lọc kiến trúc
Huấn luyện nhanh mỗi kiến trúc trên **20k ảnh train / 6k ảnh val** với cùng siêu tham số mặc định, đo thêm thời gian suy luận
**12 ô trên CPU** (máy chủ có thể không có GPU).
**Luật chọn**: trong các mô hình đạt ngân sách tốc độ, lấy mô hình **nhanh nhất** trong nhóm có F1 cách F1 tốt nhất không quá 0.2%.

In [ ]:
DEFAULT_HP = dict(lr=1e-3, weight_decay=1e-4, label_smoothing=0.05, dropout=0.2, warmup=0.05)
scr_train = train_df.sample(CFG["screen_train"], random_state=0)
scr_val = val_df.sample(CFG["screen_val"], random_state=0)
scr_tl = loader(scr_train, train_tf, shuffle=True)
scr_vl = loader(scr_val, eval_tf, workers=4)

rows = []
for arch in pl.ARCHS:
    seed_all(CFG["seed"])
    hp = dict(DEFAULT_HP, lr=3e-3 if arch == "malexnet" else 1e-3)
    t0 = time.time()
    model, hist, _ = fit(arch, hp, scr_tl, scr_vl, CFG["screen_epochs"], verbose=False, pretrained=arch != "malexnet")
    p, y, vl = pl.predict(model, scr_vl, device)
    m = pl.binary_metrics(y, p)
    rows.append(dict(arch=arch, params_M=pl.count_params(model) / 1e6, val_loss=vl, **m,
                     cpu_ms_12slots=pl.latency_ms(copy.deepcopy(model).float().cpu(), torch.device("cpu")),
                     gpu_ms_12slots=pl.latency_ms(model, device), minutes=(time.time() - t0) / 60))
    print(f"{arch:<20} acc {m['accuracy']:.4f}  f1 {m['f1']:.4f}  cpu {rows[-1]['cpu_ms_12slots']:.1f} ms")
    del model; gc.collect(); torch.cuda.empty_cache()

screen = pd.DataFrame(rows).set_index("arch").sort_values("f1", ascending=False)
screen.to_csv(RESULTS / "screening.csv")
display(screen.style.format(precision=4).background_gradient(subset=["f1", "roc_auc"], cmap="Greens"))

In [ ]:
ok = screen[screen.cpu_ms_12slots <= CFG["latency_budget_ms"]]
cand = ok if len(ok) else screen
ARCH = cand[cand.f1 >= cand.f1.max() - 0.002].sort_values("cpu_ms_12slots").index[0]
print("Kiến trúc được chọn:", ARCH)

fig, ax = plt.subplots(figsize=(7, 4.5))
for a, r in screen.iterrows():
    ax.scatter(r.cpu_ms_12slots, r.f1, s=60 + r.params_M * 25, alpha=.7,
               color="#ef4444" if a == ARCH else "#3b82f6")
    ax.annotate(f"{a}\n{r.params_M:.1f}M", (r.cpu_ms_12slots, r.f1), textcoords="offset points", xytext=(6, -4), fontsize=8)
ax.axvline(CFG["latency_budget_ms"], ls="--", c="gray", lw=1)
ax.set_xlabel("Thời gian CPU cho 12 ô (ms)"); ax.set_ylabel("F1 (val, sàng lọc)")
ax.set_title("Độ chính xác và tốc độ (cỡ điểm = số tham số)")
savefig("screening"); plt.show()

## 3. Tối ưu siêu tham số (Optuna)
- Không gian tìm: learning rate, weight decay, label smoothing, dropout, tỉ lệ warmup, batch size.
- **TPE sampler** + **MedianPruner** (dừng thử nghiệm có val loss kém hơn trung vị sau mỗi epoch).
- Mục tiêu: **val loss (cross-entropy)** — mượt và nhạy hơn accuracy khi accuracy đã ~99%.
- Thử nghiệm đầu tiên là bộ tham số mặc định, để chắc chắn kết quả tốt nhất không tệ hơn mặc định.

In [ ]:
tl_cache = {CFG["batch"]: scr_tl}
def train_loader_for(bs):
    if bs not in tl_cache:
        tl_cache[bs] = loader(scr_train, train_tf, bs=bs, shuffle=True)
    return tl_cache[bs]

def objective(trial):
    hp = dict(lr=trial.suggest_float("lr", 1e-4, 5e-3, log=True),
              weight_decay=trial.suggest_float("weight_decay", 1e-6, 5e-2, log=True),
              label_smoothing=trial.suggest_float("label_smoothing", 0.0, 0.15),
              dropout=trial.suggest_float("dropout", 0.0, 0.5),
              warmup=trial.suggest_float("warmup", 0.0, 0.15))
    bs = trial.suggest_categorical("batch", [128, 256])
    seed_all(CFG["seed"])
    model, hist, _ = fit(ARCH, hp, train_loader_for(bs), scr_vl, CFG["trial_epochs"], trial=trial, verbose=False)
    p, y, vl = pl.predict(model, scr_vl, device)
    m = pl.binary_metrics(y, p)
    trial.set_user_attr("val_acc", m["accuracy"]); trial.set_user_attr("val_f1", m["f1"])
    del model; torch.cuda.empty_cache()
    return vl

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=CFG["seed"]),
                            pruner=optuna.pruners.MedianPruner(n_startup_trials=4, n_warmup_steps=1))
study.enqueue_trial(dict(DEFAULT_HP, batch=CFG["batch"]))
t0 = time.time()
study.optimize(objective, n_trials=CFG["trials"], gc_after_trial=True)
print(f"Xong {len(study.trials)} thử nghiệm trong {(time.time() - t0) / 60:.1f} phút")
print("Tốt nhất:", study.best_value, study.best_params)

trials = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))
trials.to_csv(RESULTS / "optuna_trials.csv", index=False)
display(trials.sort_values("value").head(8))

In [ ]:
from optuna.visualization import matplotlib as ovm
fig = plt.figure(figsize=(14, 4))
ax1 = fig.add_subplot(1, 2, 1)
done = [t for t in study.trials if t.state.name == "COMPLETE"]
ax1.plot([t.number for t in done], [t.value for t in done], "o", label="thử nghiệm")
ax1.plot([t.number for t in done], np.minimum.accumulate([t.value for t in done]), "-", c="#ef4444", label="tốt nhất")
ax1.set_xlabel("thử nghiệm"); ax1.set_ylabel("val loss"); ax1.set_title("Lịch sử tối ưu"); ax1.legend()
ax2 = fig.add_subplot(1, 2, 2)
try:
    imp = optuna.importance.get_param_importances(study)
    ax2.barh(list(imp)[::-1], list(imp.values())[::-1], color="#6366f1"); ax2.set_title("Mức quan trọng siêu tham số")
except Exception as e:
    ax2.text(0.1, 0.5, f"không tính được: {e}"); ax2.axis("off")
plt.tight_layout(); savefig("optuna"); plt.show()

## 4. Huấn luyện cuối
Toàn bộ tập train (~94k ảnh) với siêu tham số tốt nhất, **EMA** trọng số, chọn epoch có val loss thấp nhất (đánh giá trên toàn bộ val ~18.6k ảnh).

In [ ]:
del scr_tl, scr_vl          # tắt worker của các DataLoader sàng lọc để giải phóng RAM
tl_cache.clear(); gc.collect()

best_hp = {k: study.best_params[k] for k in ["lr", "weight_decay", "label_smoothing", "dropout", "warmup"]}
BS = study.best_params["batch"]
print("Siêu tham số cuối:", best_hp, "batch", BS)

full_tl = loader(train_df, train_tf, bs=BS, shuffle=True)
val_l = loader(val_df, eval_tf, workers=4)
seed_all(CFG["seed"])
t0 = time.time()
model, hist, best_ep = fit(ARCH, best_hp, full_tl, val_l, CFG["final_epochs"], ema=True)
train_minutes = (time.time() - t0) / 60
print(f"Epoch tốt nhất: {best_ep} | thời gian huấn luyện {train_minutes:.1f} phút")
hist.to_csv(RESULTS / "final_history.csv", index=False)
torch.save({"arch": ARCH, "state_dict": model.state_dict(), "hp": best_hp, "img_size": CFG["img_size"]},
           RESULTS / "best_model.pth")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(hist.epoch, hist.train_loss, "o-", label="train"); ax[0].plot(hist.epoch, hist.val_loss, "o-", label="val")
ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(hist.epoch, hist.train_acc, "o-", label="train"); ax[1].plot(hist.epoch, hist.val_acc, "o-", label="val")
ax[1].plot(hist.epoch, hist.val_f1, "s--", label="val F1"); ax[1].set_title("Accuracy / F1"); ax[1].legend()
ax[2].plot(hist.epoch, hist.lr, "o-", c="#6366f1"); ax[2].set_title("Learning rate (cuối epoch)")
for a in ax:
    a.axvline(best_ep, ls="--", c="gray", lw=1); a.set_xlabel("epoch")
plt.tight_layout(); savefig("training_curves"); plt.show()

## 5. Đánh giá
### 5.1 Metric tổng hợp trên tập test (các ngày chưa từng thấy)
Ngưỡng quyết định được chọn trên **val** (tối đa F1), rồi mới áp lên test — không chọn ngưỡng trên test.

In [ ]:
test_l = loader(test_df, eval_tf, workers=4)
ab_l = loader(ab_df, eval_tf, workers=4)

p_val, y_val, _ = pl.predict(model, val_l, device)
p_test, y_test, test_loss = pl.predict(model, test_l, device)
p_tta, _, _ = pl.predict(model, test_l, device, tta=True)
THR = pl.best_threshold(y_val, p_val)
print("Ngưỡng chọn trên val:", THR)

table = pd.DataFrame({
    "test @0.5": pl.binary_metrics(y_test, p_test, 0.5),
    f"test @{THR:.2f} (ngưỡng val)": pl.binary_metrics(y_test, p_test, THR),
    "test + TTA lật ngang @0.5": pl.binary_metrics(y_test, p_tta, 0.5),
    "val @0.5": pl.binary_metrics(y_val, p_val, 0.5),
}).T
table["errors"] = [int(((p >= t) != y).sum()) for p, y, t in
                   [(p_test, y_test, .5), (p_test, y_test, THR), (p_tta, y_test, .5), (p_val, y_val, .5)]]
display(table.style.format(precision=4))
print(classification_report(y_test, (p_test >= THR).astype(int), target_names=pl.CLASSES, digits=4))

In [ ]:
pred = (p_test >= THR).astype(int)
cm = confusion_matrix(y_test, pred)
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=pl.CLASSES, yticklabels=pl.CLASSES, ax=ax[0], cbar=False)
ax[0].set_title("Ma trận nhầm lẫn (số ảnh)"); ax[0].set_xlabel("dự đoán"); ax[0].set_ylabel("thật")
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt=".4f", cmap="Blues", xticklabels=pl.CLASSES,
            yticklabels=pl.CLASSES, ax=ax[1], cbar=False)
ax[1].set_title("Chuẩn hoá theo hàng (recall từng lớp)"); ax[1].set_xlabel("dự đoán"); ax[1].set_ylabel("thật")
plt.tight_layout(); savefig("confusion_matrix"); plt.show()

### 5.2 Đường ROC và Precision–Recall

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
fpr, tpr, _ = roc_curve(y_test, p_test)
ax[0].plot(fpr, tpr, lw=2, label=f"AUC = {table.iloc[0].roc_auc:.5f}"); ax[0].plot([0, 1], [0, 1], "--", c="gray")
ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate"); ax[0].set_title("ROC"); ax[0].legend()
axins = ax[0].inset_axes([0.45, 0.15, 0.5, 0.45]); axins.plot(fpr, tpr, lw=2); axins.set_xlim(0, 0.05); axins.set_ylim(0.95, 1.0)
axins.set_title("phóng to", fontsize=8)
prec, rec, _ = precision_recall_curve(y_test, p_test)
ax[1].plot(rec, prec, lw=2, label=f"AP = {table.iloc[0].avg_precision:.5f}")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision–Recall"); ax[1].legend()
ax[1].set_ylim(0.9, 1.005)
plt.tight_layout(); savefig("roc_pr"); plt.show()

### 5.3 Hiệu chỉnh xác suất (calibration)
Xác suất đầu ra được dùng để làm mượt trạng thái ô theo thời gian, nên cần "trung thực". Đo **ECE** và dùng
**temperature scaling** (một tham số T học trên val) để hiệu chỉnh. T được lưu kèm mô hình.

In [ ]:
def to_logit(p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return np.log(p / (1 - p))

def nll(y, p):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

z_val, z_test = to_logit(p_val), to_logit(p_test)
grid = np.linspace(0.3, 4.0, 300)
TEMP = float(grid[np.argmin([nll(y_val, 1 / (1 + np.exp(-z_val / t))) for t in grid])])
p_test_cal = 1 / (1 + np.exp(-z_test / TEMP))
calib = pd.DataFrame({"ECE": [pl.expected_calibration_error(y_test, p_test), pl.expected_calibration_error(y_test, p_test_cal)],
                      "NLL": [nll(y_test, p_test), nll(y_test, p_test_cal)]}, index=["trước", f"sau (T={TEMP:.2f})"])
display(calib.style.format(precision=5))

fig, ax = plt.subplots(figsize=(5.5, 5))
for p, name in [(p_test, "trước"), (p_test_cal, "sau temperature scaling")]:
    bins = np.linspace(0, 1, 11); idx = np.digitize(p, bins[1:-1])
    xs = [p[idx == b].mean() for b in range(10) if (idx == b).sum() > 30]
    ys = [y_test[idx == b].mean() for b in range(10) if (idx == b).sum() > 30]
    ax.plot(xs, ys, "o-", label=name)
ax.plot([0, 1], [0, 1], "--", c="gray"); ax.set_xlabel("xác suất dự đoán 'có xe'"); ax.set_ylabel("tỉ lệ có xe thực tế")
ax.set_title("Biểu đồ độ tin cậy"); ax.legend(); savefig("calibration"); plt.show()

### 5.4 Theo thời tiết và theo camera

In [ ]:
res = test_df.assign(p=p_test, pred=(p_test >= THR).astype(int))
res["correct"] = res.pred == res.label
def group_metrics(g):
    m = pl.binary_metrics(g.label.values, g.p.values, THR)
    return pd.Series({"n": len(g), "accuracy": m["accuracy"], "f1": m["f1"], "errors": int((~g.correct).sum())})
by_weather = res.groupby("weather").apply(group_metrics)
by_camera = res.groupby("camera").apply(group_metrics)
display(by_weather.style.format({"accuracy": "{:.4f}", "f1": "{:.4f}", "n": "{:.0f}", "errors": "{:.0f}"}))
display(by_camera.style.format({"accuracy": "{:.4f}", "f1": "{:.4f}", "n": "{:.0f}", "errors": "{:.0f}"}))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1, 3]})
by_weather.accuracy.plot.bar(ax=ax[0], color="#14b8a6"); ax[0].set_ylim(by_weather.accuracy.min() - 0.01, 1)
ax[0].set_title("Accuracy theo thời tiết"); ax[0].tick_params(axis="x", rotation=0)
by_camera.accuracy.plot.bar(ax=ax[1], color="#6366f1"); ax[1].set_ylim(by_camera.accuracy.min() - 0.01, 1)
ax[1].set_title("Accuracy theo camera"); ax[1].tick_params(axis="x", rotation=0)
plt.tight_layout(); savefig("by_group"); plt.show()

### 5.5 Tổng quát sang camera chưa từng thấy (CNRPark A/B)
Hai camera này khác góc, khác độ cao, khác bãi — gần với tình huống lắp mô hình lên camera mới nhất có thể đo được
mà không cần dữ liệu tự thu.

In [ ]:
p_ab, y_ab, _ = pl.predict(model, ab_l, device)
ab_res = ab_df.assign(p=p_ab)
gen = pd.DataFrame({"CNR-EXT test (cùng camera, ngày mới)": pl.binary_metrics(y_test, p_test, THR),
                    "CNRPark A/B (camera mới)": pl.binary_metrics(y_ab, p_ab, THR),
                    **{f"  camera {c[-1]}": pl.binary_metrics(g.label.values, g.p.values, THR) for c, g in ab_res.groupby("camera")}}).T
display(gen.style.format(precision=4))

### 5.6 Độ bền với nhiễu ảnh
Mô phỏng điều kiện xấu của camera thật trên 4000 ảnh test: thay đổi độ sáng, nhiễu cảm biến, mất nét, độ phân giải thấp, che khuất góc ảnh.

In [ ]:
sub = test_df.sample(4000, random_state=0)
X = torch.stack([eval_tf(viewer.load(test_df.index.get_loc(i))) for i in sub.index]).to(device)
Y = torch.tensor(sub.label.values, device=device)

@torch.no_grad()
def acc_on(Xp):
    model.eval(); preds = []
    for i in range(0, len(Xp), 500):
        with torch.autocast(device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            preds.append(torch.softmax(model(Xp[i:i + 500].contiguous(memory_format=torch.channels_last)).float(), 1)[:, 1])
    return ((torch.cat(preds) >= THR).long() == Y).float().mean().item()

TESTS = {"brightness": [0.3, 0.5, 0.7, 1.0, 1.3, 1.6, 2.0], "noise": [0, 0.03, 0.06, 0.1, 0.15, 0.2],
         "blur": [0, 1, 2, 3, 4, 5], "lowres": [1, 2, 3, 4, 6, 8], "occlusion": [0, 0.2, 0.35, 0.5, 0.65]}
TITLES = {"brightness": "Hệ số độ sáng", "noise": "Độ lệch chuẩn nhiễu", "blur": "Bán kính làm mờ",
          "lowres": "Hệ số giảm độ phân giải", "occlusion": "Tỉ lệ cạnh bị che"}
robust = {k: [acc_on(pl.perturb(X, k, v)) for v in vals] for k, vals in TESTS.items()}
fig, ax = plt.subplots(1, 5, figsize=(18, 3.3), sharey=True)
for a, (k, vals) in zip(ax, TESTS.items()):
    a.plot(vals, robust[k], "o-", c="#ef4444"); a.set_title(TITLES[k]); a.set_ylim(0.5, 1.0)
ax[0].set_ylabel("accuracy")
plt.tight_layout(); savefig("robustness"); plt.show()
pd.DataFrame({k: pd.Series(v, index=[str(x) for x in TESTS[k]]) for k, v in robust.items()}).round(4)

### 5.7 Phân tích lỗi: các ảnh bị dự đoán sai tự tin nhất

In [ ]:
fp = res[(res.label == 0) & (res.pred == 1)].sort_values("p", ascending=False).head(8)
fn = res[(res.label == 1) & (res.pred == 0)].sort_values("p").head(8)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for r, (df_, name) in enumerate([(fp, "thật: TRỐNG → đoán có xe"), (fn, "thật: CÓ XE → đoán trống")]):
    for c in range(8):
        a = axes[r, c]; a.axis("off")
        if c < len(df_):
            i = df_.index[c]
            a.imshow(viewer.load(test_df.index.get_loc(i)))
            a.set_title(f"p={df_.p[i]:.2f} {df_.camera[i]}\n{df_.weather[i][:4]}", fontsize=8)
    axes[r, 0].text(-0.15, 0.5, name, transform=axes[r, 0].transAxes, rotation=90, va="center", ha="right", fontsize=9)
plt.suptitle(f"Dương tính giả: {int(((res.label == 0) & (res.pred == 1)).sum())}  |  Âm tính giả: {int(((res.label == 1) & (res.pred == 0)).sum())}")
plt.tight_layout(); savefig("errors"); plt.show()

### 5.8 Grad-CAM: mô hình nhìn vào đâu?

In [ ]:
idx = list(res[res.label == 1].sample(4, random_state=3).index) + list(res[res.label == 0].sample(4, random_state=3).index)
xb = torch.stack([eval_tf(viewer.load(test_df.index.get_loc(i))) for i in idx]).to(device)
cams, probs = pl.gradcam(model.float(), pl.cam_layer(model, ARCH), xb)
fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for c, i in enumerate(idx):
    im = pl.denorm(xb[c])
    axes[0, c].imshow(im); axes[0, c].set_title(f"{'có xe' if test_df.label[i] else 'trống'} | p={probs[c]:.2f}", fontsize=8)
    axes[1, c].imshow(im); axes[1, c].imshow(cams[c], cmap="jet", alpha=0.45)
for a in axes.ravel(): a.axis("off")
plt.suptitle("Hàng dưới: vùng ảnh ảnh hưởng mạnh tới dự đoán 'có xe'"); plt.tight_layout(); savefig("gradcam"); plt.show()

### 5.9 Tốc độ và kích thước

In [ ]:
cpu_model = copy.deepcopy(model).float().cpu().eval()
torch.save(cpu_model.state_dict(), RESULTS / "_size_probe.pth")
speed = pd.DataFrame({
    "tham số (triệu)": [pl.count_params(model) / 1e6],
    "dung lượng (MB)": [(RESULTS / "_size_probe.pth").stat().st_size / 2**20],
    "CPU 12 ô (ms)": [pl.latency_ms(cpu_model, torch.device("cpu"))],
    "CPU 1 ô (ms)": [pl.latency_ms(cpu_model, torch.device("cpu"), batch=1)],
    "GPU 12 ô (ms)": [pl.latency_ms(model, device) if device.type == "cuda" else float("nan")],
}, index=[ARCH])
(RESULTS / "_size_probe.pth").unlink()
display(speed.style.format(precision=2))

## 6. Tinh chỉnh với dữ liệu tự thu (xe mô hình)
Thu ảnh từ chính Pi Camera của bạn bằng `collect_crops.py`, ví dụ:
```
python collect_crops.py --url http://<ip-pi>:8081/snapshot.jpg --occupied 1,3,5,8 --frames 40
```
(các ô liệt kê được lưu vào `data/custom/occupied`, các ô còn lại vào `data/custom/empty`). Đổi vị trí xe, ánh sáng vài lần
rồi chạy lại cell dưới. Khi tinh chỉnh có trộn thêm 5k ảnh CNR-EXT để mô hình không quên kiến thức cũ.

In [ ]:
CUSTOM = DATA / "custom"
custom_result = None
if (CUSTOM / "empty").exists() and (CUSTOM / "occupied").exists() and len(pl.load_folder_dataset(CUSTOM)) >= 40:
    from sklearn.model_selection import train_test_split
    cdf = pl.load_folder_dataset(CUSTOM)
    ctr, cte = train_test_split(cdf, test_size=0.25, stratify=cdf.label, random_state=0)
    print("Ảnh tự thu:", cdf.label.value_counts().to_dict(), "| train", len(ctr), "test", len(cte))
    cte_l = loader(cte, eval_tf, workers=2)
    pb, yb, _ = pl.predict(model, cte_l, device)
    mix = pd.concat([ctr] * max(1, 5000 // max(len(ctr), 1)) + [train_df.sample(5000, random_state=1)])
    ft_hp = dict(best_hp, lr=best_hp["lr"] / 5, warmup=0.1)
    ft_model, ft_hist, _ = fit(ARCH, ft_hp, loader(mix, train_tf, bs=BS, shuffle=True, workers=4), cte_l, 8,
                               ema=True, init=model.state_dict())
    pa, ya, _ = pl.predict(ft_model, cte_l, device)
    p_after_test, _, _ = pl.predict(ft_model, test_l, device)
    custom_result = pd.DataFrame({"trước tinh chỉnh": pl.binary_metrics(yb, pb, THR), "sau tinh chỉnh": pl.binary_metrics(ya, pa, THR),
                                  "CNR-EXT test sau tinh chỉnh": pl.binary_metrics(y_test, p_after_test, THR)}).T
    display(custom_result.style.format(precision=4))
    model = ft_model
else:
    print("Chưa có dữ liệu tự thu trong data/custom/{empty,occupied} (cần ≥ 40 ảnh) → bỏ qua bước tinh chỉnh.")

## 7. Xuất mô hình cho `pc_server`
- `pc_server/models/occupancy.pt` — TorchScript (không cần code kiến trúc khi nạp)
- `pc_server/models/occupancy.json` — kích thước ảnh, chuẩn hoá, ngưỡng, nhiệt độ T, metric
Bật trên server: `$env:DETECTOR="classifier"`.

In [ ]:
final = copy.deepcopy(model).float().cpu().eval().to(memory_format=torch.contiguous_format)
example = torch.randn(12, 3, CFG["img_size"], CFG["img_size"])
with torch.no_grad():
    ts = torch.jit.freeze(torch.jit.trace(final, example))
    diff = (ts(example) - final(example)).abs().max().item()
ts.save(str(EXPORT / "occupancy.pt"))
print("Sai khác TorchScript vs PyTorch:", diff)

test_final = pl.binary_metrics(y_test, p_test, THR)
meta = {
    "arch": ARCH, "img_size": CFG["img_size"], "mean": pl.MEAN, "std": pl.STD, "classes": pl.CLASSES,
    "threshold": THR, "temperature": TEMP, "hyperparameters": best_hp, "batch": BS, "best_epoch": int(best_ep),
    "dataset": "CNRPark+EXT (train split theo ngày)", "finetuned_on_custom": custom_result is not None,
    "test_metrics": {k: round(float(v), 5) for k, v in test_final.items()},
    "unseen_camera_metrics": {k: round(float(v), 5) for k, v in pl.binary_metrics(y_ab, p_ab, THR).items()},
    "created": time.strftime("%Y-%m-%d %H:%M"),
}
json.dump(meta, open(EXPORT / "occupancy.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False)
json.dump({"screening": screen.reset_index().to_dict("records"), "best_params": study.best_params,
           "test": table.reset_index().to_dict("records"), "calibration": calib.reset_index().to_dict("records"),
           "by_weather": by_weather.reset_index().to_dict("records"), "by_camera": by_camera.reset_index().to_dict("records"),
           "unseen_cameras": gen.reset_index().to_dict("records"), "robustness": robust,
           "speed": speed.reset_index().to_dict("records"), "train_minutes": train_minutes},
          open(RESULTS / "metrics.json", "w", encoding="utf-8"), indent=2, ensure_ascii=False, default=float)
print("Đã lưu:", EXPORT / "occupancy.pt", "và", EXPORT / "occupancy.json")

In [ ]:
print("=" * 60)
print(f"Kiến trúc              : {ARCH}  ({pl.count_params(model) / 1e6:.2f}M tham số)")
print(f"Siêu tham số           : {best_hp} | batch {BS}")
print(f"Test CNR-EXT           : acc {test_final['accuracy']:.4f} | F1 {test_final['f1']:.4f} | AUC {test_final['roc_auc']:.5f}")
mab = pl.binary_metrics(y_ab, p_ab, THR)
print(f"Camera mới (CNRPark)   : acc {mab['accuracy']:.4f} | F1 {mab['f1']:.4f} | AUC {mab['roc_auc']:.5f}")
print(f"Hiệu chỉnh             : ECE {calib.ECE.iloc[0]:.4f} → {calib.ECE.iloc[1]:.4f} (T={TEMP:.2f})")
print(f"Tốc độ CPU (12 ô)      : {speed['CPU 12 ô (ms)'].iloc[0]:.1f} ms")
print("=" * 60)